In [4]:
import os
import math
import numpy as np
import time
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import seaborn as sns
from tqdm.auto import tqdm

In [5]:
import jax
import jax.numpy as jnp
print("Using jax", jax.__version__)

Using jax 0.10.0


In [6]:
a = jnp.zeros((2, 5), dtype=jnp.float32)
print(a)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [7]:
b = jnp.arange(6)
print(b)

[0 1 2 3 4 5]


In [8]:
b.__class__

jaxlib._jax.ArrayImpl

In [10]:
b.device

CudaDevice(id=0)

In [20]:
b_gpu = jax.device_put(b)
print(f'Device put: {b_gpu.__class__} on {b_gpu.device}')

b_cpu = jax.device_get(b)
print(f'Device get: {b_cpu.__class__} on {b_cpu.device}')

Device put: <class 'jaxlib._jax.ArrayImpl'> on cuda:0
Device get: <class 'numpy.ndarray'> on cpu


In [21]:
b_cpu + b_gpu 

Array([ 0,  2,  4,  6,  8, 10], dtype=int32)

In [23]:
jax.devices()

[CudaDevice(id=0)]

In [24]:
b_new = b.at[0].set(1)
print('Original array:', b)
print('Changed array:', b_new)

Original array: [0 1 2 3 4 5]
Changed array: [1 1 2 3 4 5]


In [25]:
rng = jax.random.PRNGKey(42)

In [26]:
# A non-desirable way of generating pseudo-random numbers...
jax_random_number_1 = jax.random.normal(rng)
jax_random_number_2 = jax.random.normal(rng)
print('JAX - Random number 1:', jax_random_number_1)
print('JAX - Random number 2:', jax_random_number_2)

# Typical random numbers in NumPy
np.random.seed(42)
np_random_number_1 = np.random.normal()
np_random_number_2 = np.random.normal()
print('NumPy - Random number 1:', np_random_number_1)
print('NumPy - Random number 2:', np_random_number_2)

JAX - Random number 1: -0.028304616
JAX - Random number 2: -0.028304616
NumPy - Random number 1: 0.4967141530112327
NumPy - Random number 2: -0.13826430117118466


In [32]:
rng, subkey1, subkey2 = jax.random.split(rng, num=3)  # We create 3 new keys
jax_random_number_1 = jax.random.normal(subkey1)
jax_random_number_2 = jax.random.normal(subkey2)
print('JAX new - Random number 1:', jax_random_number_1)
print('JAX new - Random number 2:', jax_random_number_2)

JAX new - Random number 1: -0.21089035
JAX new - Random number 2: -1.8259704


In [33]:
def simple_graph(x):
    x = x + 2
    x = x ** 2
    x = x + 3
    y = x.mean()
    return y

inp = jnp.arange(3, dtype=jnp.float32)
print('Input', inp)
print('Output', simple_graph(inp))

Input [0. 1. 2.]
Output 12.666667


In [34]:
jax.make_jaxpr(simple_graph)(inp)

{ lambda ; a:f32[3]. let
    b:f32[3] = add a 2.0:f32[]
    c:f32[3] = integer_pow[y=2] b
    d:f32[3] = add c 3.0:f32[]
    e:f32[] = reduce_sum[axes=(0,) out_sharding=None] d
    f:f32[] = div e 3.0:f32[]
  in (f,) }

In [35]:
global_list = []

# Invalid function with side-effect
def norm(x):
    global_list.append(x)
    x = x ** 2
    n = x.sum()
    n = jnp.sqrt(n)
    return n

jax.make_jaxpr(norm)(inp)

{ lambda ; a:f32[3]. let
    b:f32[3] = integer_pow[y=2] a
    c:f32[] = reduce_sum[axes=(0,) out_sharding=None] b
    d:f32[] = sqrt c
  in (d,) }